> **Public copy.** This step reads the raw Jira export and the test trackers, which stay private, so it cannot be run from this repository. The outputs below are from the original run. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E, releases such as 1.51, System tag 1 to 11 and so on), so every count is exactly as computed.


# 03 Trackers: test work items and the release timeline

**Goal:** read every monthly test-metrics tracker (Dec 2018 to Oct 2026) into one table of test-work items, put every item into a group (which vendor's release work, upgrade/infra, Programme work, other, out of scope), and build a test window for every release of every vendor.

Vendors (codes only; the key is private): Vendor A has numbered releases (1.xx), Vendor B has CI releases, and Vendors C, D and E have named versions, shown with public numbers such as Vendor C 3.0.

**Input:** the tracker workbooks in `data/trackers/`. **Output:** tables `tracker_items`, `release_windows`, `release_window_runs` and `tracker_defect_keys` in `hindsight_clean.sqlite`.

How the reader copes with eight years of changing layouts:
- Every block starts with a header row ("S No", "SN#" or "Sr. No.", with "Start Date"). Columns are matched by their names, not their position.
- Older tabs have two test-case groups (Design and Execution); newer tabs have one.
- The monthly RCA summary table under the items, and TOTAL rows, are skipped.
- A month can appear in two files (for example "Dec 2019" in the 2019 and 2020 files). Only the copy in the file of the same year is used (the canonical tab rule).
- Effort is in person-days: the PD column, or PH / 8 when PD is empty.

Known gaps: Jan to Mar 2019 and Sep 2018 are missing, and 2016 to Nov 2018 is not covered by trackers at all.
October 2026 was typed in from a screenshot (`data/trackers/manual_2026_10.csv`); November 2026 is still to come. When the 2026 workbook has these tabs, delete the manual file and re-run.

In [1]:
import sqlite3, sys
from pathlib import Path
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
WORK = ROOT / 'data' / 'private' / 'hindsight_work.sqlite'    # Jira export (read only)
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', WORK)
import hs_trackers as T
from hs_release import parse
TRACKERS = ROOT / 'data' / 'trackers'
sorted(p.name for p in TRACKERS.glob('*.xlsx'))

Reading from: data/private/hindsight_work.sqlite


['Team test metrics 2026.xlsx',
 'Programme work test metrics 2022.xlsx',
 'Environment tracker.xlsx',
 'Metrics dashboard June 2017.xlsx',
 'Team test metrics 2019.xlsx',
 'Team test metrics 2020.xlsx',
 'Team test metrics 2021.xlsx',
 'Team test metrics 2022.xlsx',
 'Team test metrics 2023.xlsx',
 'Team test metrics 2024.xlsx',
 'Team test metrics 2025.xlsx']

## 1. Read all workbooks

In [2]:
wb_rows = T.read_all(TRACKERS)
manual = T.read_manual(TRACKERS)          # months typed in from screenshots (October 2026)
print(len(manual), 'rows typed in by hand from', sorted(manual.file.unique()) if len(manual) else 'none')
raw = T.clean(pd.concat([wb_rows, manual], ignore_index=True))
print(len(raw), 'rows read;', int(raw.canonical_tab.sum()), 'in canonical tabs')
items = raw[raw.canonical_tab & (raw.row_kind == 'item')].copy()
print(len(items), 'test-work items')
items.groupby(['stream_file', 'tab_year']).size().unstack(0).fillna(0).astype(int)

10 rows typed in by hand from ['manual_2026_10.csv']
1896 rows read; 1742 in canonical tabs
1532 test-work items


stream_file,Programme work,Vendor A
tab_year,,
2018.0,0,10
2019.0,0,109
2020.0,0,186
2021.0,24,147
2022.0,4,143
2023.0,0,194
2024.0,0,245
2025.0,0,272
2026.0,0,198


## 2. Data quality checks

- Rows with no dates or no effort.
- Date typos (years outside 2015 to 2027, or an end before the start).

In [3]:
print('no start date:', int(items.start.isna().sum()))
print('no effort (person-days):', int(items.person_days.isna().sum()))
items.date_flag.value_counts(dropna=False)

no start date: 22
no effort (person-days): 15


date_flag
None                 1518
end before start       12
year out of range       2
Name: count, dtype: int64

## 3. Which release is each item for?

The release is read from the Functionality Area and Project text, with the same parser as notebook 02.
The Build column is kept separately (`build_release`), because it often names the build installed, not the release being tested.

Each item then goes into one of four groups:
- **release**: a Vendor A 1.xx or Vendor B release;
- **upgrade/infra**: Windows and security patches, database, hardware, environment and similar work (out of release scoring, kept for effort);
- **Programme work**: its own stream;
- **unclear**: everything else (for example Vendor C app work and test support).

In [4]:
items = T.tag_releases(items, parse)
items.work_group.value_counts()

work_group
unclear          555
upgrade/infra    503
release          446
Programme work              28
Name: count, dtype: int64

## 3b. Final groups (reviewed by Jagan, 23 Nov 2026)

Jagan went through every Functionality Area name (853 of them) in a review workbook. His decisions are in `data/trackers/area_review.csv`. The order used:
1. his decision for that area name;
2. agreed rules: perf and automation upkeep are out of scope, gift cards go to Other / Gift cards, Vendor E is Vendor E, Windows 10 work is shared by Vendor A, Vendor B and upgrade/infra, Vendor C patching is shared by Vendor C and upgrade/infra;
3. a release number in the text (Vendor D work inside a Vendor A release is shared by Vendor A and Vendor D);
4. keyword rules for anything left.

Shared work has its effort split equally. Other is split into sub-groups (Other system 1, Sales audit system, Store inventory system, Other system 2, Gift cards ...) for the dashboard. Out-of-scope work (perf, automation upkeep) is kept for the effort totals but never used for risk.

In [5]:
items = T.refine_groups(T.classify_items(items, TRACKERS / 'area_review.csv'))
shares = T.effort_shares(items)
shares.groupby('group').agg(items=('share', 'sum'), person_days=('person_days', 'sum')).round(1).sort_values('person_days', ascending=False)

,items,person_days
group,,
Vendor A,340.5,2136.5
Vendor B,130.5,1059.4
Upgrade/infra (EUC & environment),399.5,1013.3
Vendor C,223.5,871.6
Out of scope / Perf,92.0,611.5
Out of scope / Automation,62.0,322.2
Other / General,39.0,252.9
Other / Sales audit system,33.0,116.5
Programme work,41.0,115.5


## 4. Release test windows

Releases come in two kinds:
- **numbered releases**: Vendor A 1.xx (Vendor A) and CI n (Vendor B), read from the tracker text;
- **named programmes** for the vendors without numbered releases: Vendor C, Vendor D and Vendor E versions, shown with public numbers (Vendor C 3.0 to 5.4, Vendor D v2.24 to v2.29, Vendor E 6.0 to 8.2). The programme list, with the months each one ran, is in `scripts/hs_programmes.py`.

Dates are first kept within one month of the tab they were reported in (this removes typos like 2027).
A release's months are then grouped into runs; the run with the most effort is its main window. Later runs (for example a fix drop a year later) are kept in `release_window_runs` and used as extra candidate windows when matching defects by date. Out-of-scope work (perf, automation upkeep) does not stretch a window.

In [6]:
rel_work = T.release_work(items)
windows = T.release_windows(rel_work)
runs = T.window_runs(rel_work)
show = windows.copy()
for c in ['window_start', 'window_end', 'exec_start']:
    show[c] = show[c].dt.date
show[['release_stream', 'release_key', 'vendor', 'window_start', 'window_end', 'window_days', 'items', 'person_days', 'tc_executed', 'tracker_defects', 'stray_items_outside_window']]

,release_stream,release_key,vendor,window_start,window_end,window_days,items,person_days,tc_executed,tracker_defects,stray_items_outside_window
0,Vendor A,1.37,Vendor A,2018-12-05,2018-12-31,27,2,46.000,840.0,0.0,3
1,Vendor A,1.39,Vendor A,2019-04-01,2019-12-31,275,44,273.000,7637.0,138.0,2
2,Vendor C,Vendor C 3.0,Vendor C,2019-04-29,2019-08-01,95,6,5.125,121.0,17.0,0
3,Vendor B,CI 2,Vendor B,2019-07-22,2019-08-30,40,4,35.000,1393.0,36.0,0
4,Vendor C,Vendor C 4.0,Vendor C,2019-10-01,2020-01-24,116,5,57.000,132.0,16.0,0
5,Vendor A,1.41,Vendor A,2020-01-13,2020-04-30,109,16,80.250,1216.0,40.0,9
6,Vendor B,CI 3,Vendor B,2020-03-02,2020-11-11,255,7,31.000,222.0,15.0,1
7,Vendor C,Vendor C 4.1,Vendor C,2020-05-01,2020-07-23,84,8,31.750,726.0,25.0,0
8,Vendor C,Vendor C 4.2,Vendor C,2020-08-03,2020-11-30,120,22,38.500,3305.0,49.0,0
9,Vendor E,Vendor E 6.0,Vendor E,2020-08-18,2020-11-12,87,3,3.000,65.0,3.0,0


In [7]:
windows = windows.sort_values('window_start').reset_index(drop=True)
windows['release_order'] = range(1, len(windows) + 1)                         # all vendors, by start date
windows['vendor_release_no'] = windows.groupby('vendor').cumcount() + 1       # 1st, 2nd ... release of each vendor
print(len(windows), 'releases;', len(runs), 'window runs')
windows.groupby('vendor').release_key.count().to_frame('releases')

53 releases; 68 window runs


,releases
vendor,
Vendor A,18
Vendor B,14
Vendor C,12
Vendor D,4
Vendor E,5


## 5. Defect keys written in the trackers

Testers wrote Jira keys (PROJ-####) in the Defect ID and Comments columns. Next to a release, these are known answers: they tell us which release a defect belongs to, and they let us check notebook 02.

In [8]:
keys = T.tracker_defect_keys(items)
print(len(keys), 'key mentions;', keys.key.nunique(), 'distinct defects')
with sqlite3.connect(CLEAN) as cc:
    rel = pd.read_sql('select key, release_main, confidence from defect_release', cc)
chk = keys[keys.work_group == 'release'].merge(rel, on='key', how='left')
both = chk[chk.release_main.notna()]
agree = (both.release_key == both.release_main).mean()
print(f'Tracker and Jira text agree on the release for {agree:.0%} of {len(both)} defects')

637 key mentions; 592 distinct defects
Tracker and Jira text agree on the release for 92% of 274 defects


## 6. Save

In [9]:
save = items.copy()
for c in ['start', 'end']:
    save[c] = save[c].dt.strftime('%Y-%m-%d')
w = windows.copy()
for c in ['window_start', 'window_end', 'exec_start']:
    w[c] = w[c].dt.strftime('%Y-%m-%d')
rr = runs.copy()
for c in ['run_start', 'run_end']:
    rr[c] = rr[c].dt.strftime('%Y-%m-%d')
keep_cols = ['file', 'tab', 'excel_row', 'tab_year', 'tab_month', 'stream_file', 'block', 'sn', 'area', 'project',
             'testing_type_col', 'phase', 'start', 'end', 'tc_overall', 'tc_executed', 'person_days', 'pd', 'ph',
             'defect_count', 'defect_ids', 'defect_rca', 'build', 'comments', 'date_flag', 'work_group',
             'release_stream', 'release_main', 'release_key', 'ci_version', 'inc', 'ptf', 'build_release',
             'final_group', 'top_group', 'group_source']
with sqlite3.connect(CLEAN) as cc:
    save[[c for c in keep_cols if c in save]].to_sql('tracker_items', cc, if_exists='replace', index=False)
    w.to_sql('release_windows', cc, if_exists='replace', index=False)
    rr.to_sql('release_window_runs', cc, if_exists='replace', index=False)
    keys.to_sql('tracker_defect_keys', cc, if_exists='replace', index=False)
    shares.to_sql('tracker_effort_shares', cc, if_exists='replace', index=False)
print('saved')

saved
